<a href="https://colab.research.google.com/github/Ibrah-N/Deep-Learning-Projects-Computer-Vision/blob/main/Pytorch_4_rnn_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import torch
print(torch.__version__)

2.10.0+cpu


# Data Preparation

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/Ibrah-N/Deep-Learning-Projects-Computer-Vision/main/100_Unique_QA_Dataset.csv"
df = pd.read_csv(url, header=0)

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
def tokenize(text):
  text = text.lower()
  text = text.replace(",", "")
  text = text.replace(".", "")
  text = text.replace("'", "")
  text = text.replace("?", "")
  return text.split()

In [4]:
tokenize("What is your name in '1990'?")

['what', 'is', 'your', 'name', 'in', '1990']

In [5]:
question_dataset_vocab = {"UNKNOWN": 0}


def build_vocab(row):

  question = row["question"]
  answer = row['answer']

  question_tokens = tokenize(question)
  answer_tokens = tokenize(answer)

  combine_tokens = question_tokens + answer_tokens

  for token in combine_tokens:
    if token not in question_dataset_vocab:
      question_dataset_vocab[token] = len(question_dataset_vocab)

In [6]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
print("Samples..")
print("*"*20)
for idx, (k, v) in enumerate(question_dataset_vocab.items()):
  print(k," --> ", v)

  if idx==5:
    break

print()
print("*"*20)
print("Length of Vocabulary {}".format(len(question_dataset_vocab)))

Samples..
********************
UNKNOWN  -->  0
what  -->  1
is  -->  2
the  -->  3
capital  -->  4
of  -->  5

********************
Length of Vocabulary 324


In [8]:
def text_to_index(text, vocab):
  tokens = tokenize(text)

  indecies_list = []
  for token in tokens:
    if token in vocab:
      indecies_list.append(vocab[token])
    else:
      indecies_list.append(vocab["UNKNOWN"])

  return indecies_list

In [9]:
text_to_index("What is the capital of france", question_dataset_vocab)

[1, 2, 3, 4, 5, 6]

In [10]:
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab


  def __len__(self):
    return self.df.shape[0]


  def __getitem__(self, index):
    question_ = self.df.iloc[index]['question']
    answer_ = self.df.iloc[index]['answer']

    question_indecis = text_to_index(question_, self.vocab)
    answer_indecis = text_to_index(answer_, self.vocab)

    return torch.tensor(question_indecis), torch.tensor(answer_indecis)


In [11]:
dataset = QADataset(df, question_dataset_vocab)

In [12]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Building Model

In [17]:
class CustomModel(torch.nn.Module):

  def __init__(self, vocab, embedding_dim, hidden_dim):
    super(CustomModel, self).__init__()

    self.vocab = vocab
    self.embedding_dim = embedding_dim
    self.hidden_dim = hidden_dim
    self.vocab_size = len(vocab)



    self.embedding_layer = torch.nn.Embedding(self.vocab_size, self.embedding_dim)
    self.rnn_layer = torch.nn.RNN(self.embedding_dim, hidden_dim, batch_first=True)
    self.linear_layer = torch.nn.Linear(hidden_dim, self.vocab_size)



  def forward(self, x):
    x = self.embedding_layer(x)
    _, x = self.rnn_layer(x)
    x = self.linear_layer(x.squeeze(0))

    return x

In [18]:
question_model = CustomModel(question_dataset_vocab, embedding_dim=50, hidden_dim=100)
question_model

CustomModel(
  (embedding_layer): Embedding(324, 50)
  (rnn_layer): RNN(50, 100, batch_first=True)
  (linear_layer): Linear(in_features=100, out_features=324, bias=True)
)

In [19]:
EPOCHS = 20
LEARNING_RATE = 0.001
LOSS = torch.nn.CrossEntropyLoss()
OPTIMIZER = torch.optim.Adam(question_model.parameters(), lr=LEARNING_RATE)

In [23]:
for epoch in range(EPOCHS):

  total_loss = 0
  for batch_q, batch_a in dataloader:
    pred = question_model(batch_q)

    loss = LOSS(pred, batch_a.squeeze(0))

    OPTIMIZER.zero_grad()
    loss.backward()

    OPTIMIZER.step()

    total_loss += loss


  print(f"Epoch: {epoch + 1}, Loss {total_loss}")

Epoch: 1, Loss 522.2054443359375
Epoch: 2, Loss 421.5631408691406
Epoch: 3, Loss 327.3731994628906
Epoch: 4, Loss 253.0930633544922
Epoch: 5, Loss 184.16281127929688
Epoch: 6, Loss 126.03910827636719
Epoch: 7, Loss 83.59046936035156
Epoch: 8, Loss 55.2840576171875
Epoch: 9, Loss 38.10854721069336
Epoch: 10, Loss 26.822917938232422
Epoch: 11, Loss 20.073244094848633
Epoch: 12, Loss 15.338618278503418
Epoch: 13, Loss 12.223389625549316
Epoch: 14, Loss 9.857905387878418
Epoch: 15, Loss 8.125470161437988
Epoch: 16, Loss 6.831085681915283
Epoch: 17, Loss 5.838388442993164
Epoch: 18, Loss 5.041486740112305
Epoch: 19, Loss 4.392541885375977
Epoch: 20, Loss 3.878939628601074


# Model Testing

In [ ]:
index_to_word = {v: k for k, v in question_dataset_vocab.items()}
print(index_to_word)

In [54]:
input_x = "Which fruit is known as the king of fruits?"

# 2 Convert to indices
indexed_x = text_to_index(input_x, question_dataset_vocab)

# 3 Convert to tensor
model_input = torch.tensor(indexed_x).unsqueeze(0)

# 4 Evaluation mode
question_model.eval()

# 5 Disable gradients
with torch.no_grad():

    pred = question_model(model_input)

    # logits → probabilities
    probs = torch.softmax(pred, dim=1)

    # best class index
    predicted_index_ = torch.argmax(probs, dim=1)
# convert tensor to python int
predicted_index = predicted_index_.item()

print("Input:", input_x)
print("Tokens:", tokenized_x)
print("Indices:", indexed_x)
print("*"*30)
print("Prediction index:", predicted_index)
print("Probability:", probs[0][predicted_index])
print("Prediction: ", index_to_word[predicted_index_.item()])

Input: Which fruit is known as the king of fruits?
Tokens: ['what', 'is', 'the', 'capital', 'of', 'france']
Indices: [42, 318, 2, 62, 63, 3, 319, 5, 320]
******************************
Prediction index: 321
Probability: tensor(0.9624)
Prediction:  mango


In [51]:
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango
